# RAG 에이전트 평가 노트북 (강사님 코드 기반)
LangChain + Chroma + CacheBackedEmbeddings + Kiwi BM25 하이브리드 + `create_agent(rag_tool)` 구조 그대로.

**사용법**: 1~4 한 번 실행 → 5에서 `run('이름', 바꿀항목=값)` 으로 **한 번에 하나씩** 실험 → 6에서 비교.

In [ ]:
!pip install -qU langchain langchain-openai langchain-community "langchain-chroma>=0.1.2" langchain-hwp-hwpx-loader pdfplumber rank_bm25 kiwipiepy pandas

In [ ]:
# 1) 설정 -------------------------------------------------------------
import os, glob, time, hashlib, re, unicodedata
import pandas as pd, numpy as np
try:
    from google.colab import userdata; os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    os.environ.setdefault("OPENAI_API_KEY", "")   # 직접 입력

DOC_DIR = "/content/docs"           # 문서 폴더 (pdf/hwp/hwpx/txt/md)
Q_FILE  = "/content/questions.csv"  # 컬럼: question, answer
RESULTS = "/content/results.csv"

# 문서 주제에 맞게 문서 확인 후 수정 (에이전트가 도구를 언제 쓸지 판단하는 근거)
TOOL_DESC = "업로드된 문서에서 질문과 관련된 내용을 검색하는 도구. 문서 내용과 관련된 모든 질문에 사용하세요."
SYSTEM_PROMPT = """당신은 제공된 문서 기반 질의응답 전문가입니다.
질문이 들어오면 반드시 rag_tool을 호출해 관련 정보를 검색한 뒤, 검색된 문맥만 근거로 간결하고 정확하게 답변하세요.
문맥에 근거가 없으면 지어내지 말고 모른다고 답변하세요."""

BASE = dict(
    chunk_size=500, overlap=50,
    preprocess=False,          # 강사님 clean_korean_pdf_text 전처리
    k=4, search_type="similarity",   # "similarity" | "mmr"
    hybrid=False, bm25_k=4,    # Kiwi BM25 + 벡터 합치기
    rel_filter=False,          # LLM 문서 관련성 필터 (느려짐)
    embed_model="text-embedding-3-large",
    agent_model="gpt-5.4-mini", judge_model="gpt-5.4-mini",
)

In [ ]:
# 2) 로드 (+ 선택 전처리) ---------------------------------------------------
from langchain_core.documents import Document
from langchain_community.document_loaders import PDFPlumberLoader, TextLoader
from langchain_hwp_hwpx import HwpHwpxLoader

def load_docs():
    docs = []
    for p in sorted(glob.glob(f"{DOC_DIR}/**/*", recursive=True)):
        ext = p.lower().rsplit(".", 1)[-1]
        try:
            if ext == "pdf": docs += PDFPlumberLoader(p).load()
            elif ext in ("hwp", "hwpx"): docs += HwpHwpxLoader(p, mode="elements").load()
            elif ext in ("txt", "md"): docs += TextLoader(p, encoding="utf-8").load()
        except Exception as e: print("skip", p, e)
    return docs
RAW_DOCS = load_docs(); print(len(RAW_DOCS), "docs loaded")

# --- 강사님 전처리 (축약) ---
PAGE_NUMBER_RE = re.compile(r"^-?\s*\d+\s*-?$"); DOT_LEADER_RE = re.compile(r"\.{5,}\s*\d+$")
PRIVATE_USE_RE = re.compile(r"[-]"); INVISIBLE_RE = re.compile(r"[​-‏‪-‮]")
def clean_text(text):
    text = unicodedata.normalize("NFKC", text or "")
    text = PRIVATE_USE_RE.sub("\n- ", INVISIBLE_RE.sub("", text)).replace("ㆍ", "- ").replace("·", "- ")
    lines = [re.sub(r"[ \t]+", " ", l).strip() for l in text.replace("\r", "\n").split("\n")]
    out = []
    for l in lines:
        if not l or PAGE_NUMBER_RE.fullmatch(l) or DOT_LEADER_RE.search(l): continue
        if out and not re.search(r"(다\.|요\.|함\.|됨\.|음\.|[.!?])$", out[-1]):
            out[-1] += ("" if re.search(r"[가-힣]$", out[-1]) and re.match(r"^[가-힣]", l) else " ") + l
        else: out.append(l)
    return re.sub(r"\n{3,}", "\n\n", "\n".join(out)).strip()
def preprocess(docs):
    new = [Document(page_content=clean_text(d.page_content), metadata=dict(d.metadata)) for d in docs]
    return [d for d in new if len(d.page_content) >= 10]

In [ ]:
# 3) 분할 + 임베딩(캐시) + Chroma, Kiwi BM25 ---------------------------------
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi

kiwi = Kiwi()
ALLOWED_TAGS = {"NNG", "NNP", "NNB", "VV", "VA", "XR", "SL", "SN"}
def korean_tokenizer(text): return [t.form for t in kiwi.tokenize(text) if t.tag in ALLOWED_TAGS]

_store = {}
def build_store(cfg):
    key = hashlib.md5(f"{cfg['chunk_size']}{cfg['overlap']}{cfg['preprocess']}{cfg['embed_model']}".encode()).hexdigest()[:10]
    if key in _store: return _store[key]
    docs = preprocess(RAW_DOCS) if cfg["preprocess"] else RAW_DOCS
    splits = RecursiveCharacterTextSplitter(chunk_size=cfg["chunk_size"], chunk_overlap=cfg["overlap"],
                                            separators=["\n\n", "\n", " ", ""]).split_documents(docs)
    emb = CacheBackedEmbeddings.from_bytes_store(OpenAIEmbeddings(model=cfg["embed_model"]), LocalFileStore("./cache/"))
    db = Chroma.from_documents(splits, emb, persist_directory=f"./chroma_{key}", collection_metadata={"hnsw:space": "cosine"})
    _store[key] = (db, splits); print(f"index {key}: {len(splits)} chunks")
    return _store[key]

In [ ]:
# 4) 검색 도구 + 에이전트 -----------------------------------------------------
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent

def make_agent(cfg):
    db, splits = build_store(cfg)
    vec = db.as_retriever(search_type=cfg["search_type"], search_kwargs={"k": cfg["k"]})
    bm25 = BM25Retriever.from_documents(splits, preprocess_func=korean_tokenizer); bm25.k = cfg["bm25_k"]
    grader = init_chat_model(cfg["judge_model"]) if cfg["rel_filter"] else None

    def retrieve(q):
        docs = vec.invoke(q)
        if cfg["hybrid"]:
            seen, merged = set(), []
            for d in bm25.invoke(q) + docs:
                if d.page_content.strip() not in seen: seen.add(d.page_content.strip()); merged.append(d)
            docs = merged
        if grader:
            prompts = [[HumanMessage(f"질문: {q}\n문서: {d.page_content}\n문서가 질문에 답하는 데 관련 있으면 yes, 아니면 no만 출력.")] for d in docs]
            docs = [d for d, r in zip(docs, grader.batch(prompts)) if "yes" in r.content.lower()]
        return docs

    @tool("rag_tool", description=TOOL_DESC)
    def rag_tool(query: str) -> str:
        docs = retrieve(query)
        if not docs: return "<문맥 정보>관련 문서를 찾지 못했습니다.</문맥 정보>"
        return "<문맥 정보>" + "\n\n".join(d.page_content for d in docs) + "</문맥 정보>"

    return create_agent(model=cfg["agent_model"], tools=[rag_tool], system_prompt=SYSTEM_PROMPT)

def ask(agent, q):
    return agent.invoke({"messages": [{"role": "user", "content": q}]})["messages"][-1].content

In [ ]:
# 5) 평가: 질문당 3회, 평균시간(에이전트 호출 구간만), 3회 중 2회 이상 정답이면 통과 ---
judge_llm = None
def judge(q, gold, pred, model):
    global judge_llm
    judge_llm = judge_llm or init_chat_model(model)
    r = judge_llm.invoke([HumanMessage(f"질문: {q}\n정답: {gold}\n모델 답변: {pred}\n\n모델 답변이 정답의 핵심 내용과 일치하면 1, 아니면 0만 출력.")])
    return r.content.strip().startswith("1")

LAST = None
def run(name, **override):
    global LAST
    cfg = {**BASE, **override}
    agent = make_agent(cfg)                       # 인덱싱/에이전트 생성은 시간 측정 제외
    qs = pd.read_csv(Q_FILE); has_gold = "answer" in qs; rows = []
    for _, r in qs.iterrows():
        ts, preds = [], []
        for _ in range(3):
            t = time.perf_counter(); preds.append(ask(agent, r.question)); ts.append(time.perf_counter() - t)
        oks = [judge(r.question, r.answer, p, cfg["judge_model"]) for p in preds] if has_gold else []
        rows.append(dict(q=r.question, preds=preds, oks=oks, t=np.mean(ts)))
    LAST = pd.DataFrame(rows)
    s = dict(name=name, avg_time=round(LAST.t.mean(), 2),
             pass_rate=round(LAST.oks.map(lambda o: sum(o) >= 2).mean(), 3) if has_gold else None,
             avg_correct_runs=round(LAST.oks.map(sum).mean(), 2) if has_gold else None,
             **{k: cfg[k] for k in BASE if cfg[k] != BASE[k]})   # 바뀐 항목만 기록
    pd.DataFrame([s]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    print(s); return LAST

# 실험 예 (한 번에 하나만 변경)
# run("baseline")
# run("mmr", search_type="mmr")
# run("hybrid", hybrid=True)
# run("preprocess", preprocess=True)
# run("k6", k=6)
# run("relfilter", rel_filter=True)

In [ ]:
# 6) 비교 / 실패 질문 확인 ---------------------------------------------------
display(pd.read_csv(RESULTS))
# LAST[LAST.oks.map(lambda o: sum(o) < 2)][["q","preds"]]   # 틀린 질문 -> 검색 문제인지 생성 문제인지 확인 후 다음 개선 결정